In [ ]:
#!/usr/bin/env python3
"""
extract_transporters_notebook.py
==================================

Notebook-friendly version of the original extract_dmt_transporters.py --
same logic, just with plain Python variables instead of --command-line
flags (argparse breaks under Jupyter's %run / ipykernel_launcher.py).

Purpose
-------
You have, for ~80 Staphylococcus/Mammaliicoccus genomes, two parallel sets of files:

  1. "TransAAP outputs from each genome" -- one *.transporter.txt per genome.
     Tab-delimited, 10 columns, no header. Column 1 is the locus tag, column 2
     is the genome name (binomial OR an assembly accession like GCF_xxx),
     column 3 is the transporter family code (e.g. "DMT", "MFS", "ABC"...).

  2. "FAA files of all transporters" -- one *_transaap_transporter_load.faa per
     genome. Standard FASTA, headers are bare locus tags (e.g. ">KJGELL_00970"),
     matching column 1 of the corresponding .transporter.txt file.

A third file, map_assembly_to_binomial.txt, maps assembly accessions
(GCA_/GCF_...) to their correct full binomial species name, since some
genomes are only labeled with an accession instead of a proper species name.

What this script does
----------------------
For every genome:
  1. Parse the .transporter.txt file and pull out every locus tag whose
     family code (column 3) matches the target family (e.g. "DMT", "MFS").
  2. Look up the correct binomial species name via map_assembly_to_binomial.txt.
  3. Pull the matching protein sequences out of the genome's .faa file.
  4. Write all matching proteins into ONE combined FASTA file:
         all_genomes_<FAMILY>_combined.fasta
     Each record header is: <locustag>_<Binomial_name>_<FAMILY>
  5. Also writes a TSV summary (hit count per genome).

USAGE (Jupyter Notebook)
-------------------------
    %run extract_transporters_notebook.py

Edit the variables in the "RUN IT FROM HERE" section near the bottom of
this file BEFORE running -- especially FAMILY, which controls which
transporter family gets extracted (e.g. "DMT", "MFS", "ABC", "APC"...).
"""

import csv
import sys
from pathlib import Path


# --------------------------------------------------------------------------- #
# Parsing helpers
# --------------------------------------------------------------------------- #

def load_assembly_to_binomial_map(map_file: str) -> dict:
    """
    Parse map_assembly_to_binomial.txt.
    Returns dict mapping genome_key -> clean "Genus_species" string.
    """
    mapping = {}
    with open(map_file, "r", encoding="utf-8") as fh:
        reader = csv.reader(fh, delimiter="\t")
        for row in reader:
            if not row or not row[0].strip():
                continue
            key = row[0].strip()
            if len(row) < 3 or not row[2].strip():
                continue
            raw_name = row[2].strip()
            mapping[key] = clean_binomial(raw_name)
    return mapping


def clean_binomial(raw_name: str) -> str:
    """
    Reduce a messy species name field like:
        "Staphylococcus taiwanensis strain NTUH-S172 chromosome"
    down to a clean, filename-safe "Genus_species".
    """
    tokens = raw_name.split()
    if len(tokens) >= 2:
        genus_species = f"{tokens[0]}_{tokens[1]}"
    elif tokens:
        genus_species = tokens[0]
    else:
        genus_species = "Unknown_species"
    return sanitize_for_filename(genus_species)


def sanitize_for_filename(text: str) -> str:
    """Keep filenames safe: letters, digits, underscore, hyphen, dot only."""
    safe = []
    for ch in text:
        if ch.isalnum() or ch in ("_", "-", "."):
            safe.append(ch)
        else:
            safe.append("_")
    cleaned = "_".join(filter(None, "".join(safe).split("_")))
    return cleaned or "Unknown_species"


def parse_transporter_file(path: str, family: str):
    """
    Parse one *.transporter.txt file. Returns ([(locus_tag, genome_key,
    family_code), ...], genome_label) for rows matching `family` exactly
    (case-insensitive).
    """
    hits = []
    genome_key = None
    with open(path, "r", encoding="utf-8") as fh:
        reader = csv.reader(fh, delimiter="\t")
        for row in reader:
            if not row or len(row) < 3:
                continue
            locus_tag = row[0].strip()
            genome_in_row = row[1].strip()
            family_code = row[2].strip()
            if genome_in_row:
                genome_key = genome_in_row
            if not locus_tag or not family_code:
                continue
            if family_code.lower() == family.lower():
                hits.append((locus_tag, genome_in_row, family_code))
    return hits, genome_key


def parse_faa(path: str) -> dict:
    """Parse a FASTA (.faa) file into {locus_tag: sequence}."""
    seqs = {}
    current_id = None
    current_seq_chunks = []

    def flush():
        if current_id is not None:
            seqs[current_id] = "".join(current_seq_chunks)

    with open(path, "r", encoding="utf-8") as fh:
        for line in fh:
            line = line.rstrip("\n").rstrip("\r")
            if not line:
                continue
            if line.startswith(">"):
                flush()
                header = line[1:].strip()
                current_id = header.split()[0] if header else None
                current_seq_chunks = []
            else:
                current_seq_chunks.append(line.strip())
        flush()
    return seqs


# --------------------------------------------------------------------------- #
# File discovery
# --------------------------------------------------------------------------- #

def find_transporter_files(transaap_dir: str) -> dict:
    out = {}
    for p in Path(transaap_dir).rglob("*.transporter.txt"):
        out[p.name[: -len(".transporter.txt")]] = str(p)
    return out


def find_faa_files(faa_dir: str) -> dict:
    out = {}
    suffix = "_transaap_transporter_load.faa"
    for p in Path(faa_dir).rglob("*.faa"):
        name = p.name
        basename = name[: -len(suffix)] if name.endswith(suffix) else name[: -len(".faa")]
        out[basename] = str(p)
    return out


# --------------------------------------------------------------------------- #
# Main extraction logic
# --------------------------------------------------------------------------- #

def run(transaap_dir, faa_dir, map_file, outdir, family):
    transporter_files = find_transporter_files(transaap_dir)
    faa_files = find_faa_files(faa_dir)
    accession_to_binomial = load_assembly_to_binomial_map(map_file)

    genome_keys = sorted(set(transporter_files) | set(faa_files))
    missing_pairs = [g for g in genome_keys if g not in transporter_files or g not in faa_files]
    if missing_pairs:
        print(f"WARNING: {len(missing_pairs)} genome(s) missing a matching file in one of the two folders:", file=sys.stderr)
        for g in missing_pairs:
            print(f"    {g}", file=sys.stderr)

    outdir_path = Path(outdir).resolve()
    outdir_path.mkdir(parents=True, exist_ok=True)
    print(f"Output folder ready: {outdir_path}")

    combined_fasta_path = outdir_path / f"all_genomes_{family}_combined.fasta"

    total_sequences_written = 0
    total_sequences_missing_in_faa = 0
    genomes_processed = 0
    summary_rows = []

    with open(combined_fasta_path, "w", encoding="utf-8") as combined_fh:
        for genome_key in genome_keys:
            if genome_key in missing_pairs:
                continue

            hits, genome_label = parse_transporter_file(transporter_files[genome_key], family)
            if not hits:
                continue

            binomial = (
                accession_to_binomial.get(genome_key)
                or (clean_binomial(genome_label.replace("_", " ")) if genome_label else None)
                or sanitize_for_filename(genome_key)
            )

            sequences = parse_faa(faa_files[genome_key])
            genomes_processed += 1
            hits_written = 0

            for locus_tag, _genome_in_row, _family_code in hits:
                seq = sequences.get(locus_tag)
                if seq is None:
                    print(f"WARNING: locus tag {locus_tag} (genome {genome_key}) not found in FAA file", file=sys.stderr)
                    total_sequences_missing_in_faa += 1
                    continue

                fasta_record = f">{locus_tag}_{binomial}_{family}\n{seq}\n"
                combined_fh.write(fasta_record)

                total_sequences_written += 1
                hits_written += 1

            summary_rows.append((genome_key, binomial, hits_written))

    print(f"\nFamily searched: {family}")
    print(f"Genomes scanned: {len(genome_keys)} (skipped {len(missing_pairs)} unmatched)")
    print(f"Genomes with at least one {family} hit: {genomes_processed}")
    print(f"Total {family} sequences written: {total_sequences_written}")
    if total_sequences_missing_in_faa:
        print(f"Locus tags listed but NOT found in FAA: {total_sequences_missing_in_faa}")
    print(f"\nCombined FASTA (only file, no individual per-sequence files) written to: {combined_fasta_path}")

    summary_path = outdir_path / f"{family}_summary.tsv"
    with open(summary_path, "w", encoding="utf-8") as sfh:
        sfh.write("genome_key\tbinomial_name\tnum_hits\n")
        for genome_key, binomial, n in summary_rows:
            sfh.write(f"{genome_key}\t{binomial}\t{n}\n")
    print(f"Per-genome hit-count summary written to: {summary_path}")


# --------------------------------------------------------------------------- #
# ===========================  RUN IT FROM HERE  =========================== #
# --------------------------------------------------------------------------- #

# ---- EDIT THESE ---- #
TRANSAAP_DIR = "TransAAP outputs from each genome"   # folder with *.transporter.txt files
FAA_DIR = "FAA files of all transporters"             # folder with *.faa files
MAP_FILE = "map_assembly_to_binomial.txt"             # the accession->binomial map file
OUTDIR = "transporter_output"                         # where results get written
FAMILY = "SSPTS"                                        # <-- CHANGE THIS to "MFS", "ABC", "APC", etc.
# --------------------- #

# ---- PRE-FLIGHT CHECK ---- #
import os as _os

print("=== PRE-FLIGHT CHECK ===")
print("Current working directory:", _os.getcwd())

_problems = []

if not _os.path.isdir(TRANSAAP_DIR):
    _problems.append(f"TRANSAAP_DIR not found: {TRANSAAP_DIR!r}")
else:
    _n = sum(1 for _r, _d, _f in _os.walk(TRANSAAP_DIR) for _x in _f if _x.endswith(".transporter.txt"))
    print(f"TRANSAAP_DIR found. .transporter.txt files inside: {_n}")
    if _n == 0:
        _problems.append(f"TRANSAAP_DIR exists but contains ZERO *.transporter.txt files.")

if not _os.path.isdir(FAA_DIR):
    _problems.append(f"FAA_DIR not found: {FAA_DIR!r}")
else:
    _n2 = sum(1 for _r, _d, _f in _os.walk(FAA_DIR) for _x in _f if _x.endswith(".faa"))
    print(f"FAA_DIR found. .faa files inside: {_n2}")
    if _n2 == 0:
        _problems.append(f"FAA_DIR exists but contains ZERO *.faa files.")

if not _os.path.isfile(MAP_FILE):
    _problems.append(f"MAP_FILE not found: {MAP_FILE!r}")
else:
    print(f"MAP_FILE found: {MAP_FILE}")

if _problems:
    print("\n!!! PROBLEMS FOUND -- fix these before continuing !!!")
    for _p in _problems:
        print("  -", _p)
    print("\nrun() will NOT be called. Fix the problem(s) above, then re-run this cell.")
else:
    print(f"All checks passed. Extracting family '{FAMILY}'...\n")
    run(
        transaap_dir=TRANSAAP_DIR,
        faa_dir=FAA_DIR,
        map_file=MAP_FILE,
        outdir=OUTDIR,
        family=FAMILY,
    )